# PMC Miner
DOI-based mining the paper data through PMC: DOIs from `notebook/test_dois.csv`.

In [1]:
from pathlib import Path
import json

from pmc_miner import SearchBasedMiner, DOIBasedMiner
from pmc_miner.utils.logging import setup_logging

setup_logging()

NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
print("notebook dir :", NOTEBOOK_DIR)
print("project root :", PROJECT_ROOT)

notebook dir : /Users/yaochenr/project/pmc_data_mining/notebook
project root : /Users/yaochenr/project/pmc_data_mining


## DOI-based mining

By using DOI-based mining, you only need to specify the DOI list. For papers not in PMC open access, they will be skipped and logged to `/dois_notin_pmc.txt`.

In [2]:
doi_csv = '/Users/yaochenr/project/pmc_data_mining/notebook/test_dois.csv'
doi_out = NOTEBOOK_DIR / "doi_mining_test"

doi_miner = DOIBasedMiner(
    output_dir=str(doi_out),
    paper_type="test", # a label
    download_images=True,
)

doi_stats = doi_miner.mine_from_csv(str(doi_csv))
doi_stats

2026-05-11 15:20:50,353 - INFO - Loaded 8 DOIs from /Users/yaochenr/project/pmc_data_mining/notebook/test_dois.csv
2026-05-11 15:20:50,354 - INFO - DOI-based mining (test): 8 DOIs
2026-05-11 15:20:50,354 - INFO - [1/8] 10.1039/d2md00199c
2026-05-11 15:20:52,478 - INFO - Found PMC ID PMC9749924 for DOI 10.1039/d2md00199c (Open Access)
2026-05-11 15:20:53,002 - INFO - Retrieved metadata for 1 papers
2026-05-11 15:20:53,731 - INFO - Successfully retrieved XML for PMC9749924
2026-05-11 15:20:53,749 - INFO - Saved metadata for PMCPMC9749924
2026-05-11 15:20:53,750 - INFO - Saved XML for PMCPMC9749924
2026-05-11 15:20:53,750 - INFO - Starting supplementary download for PMCPMC9749924
2026-05-11 15:20:53,751 - INFO - Starting supplementary download for PMC9749924
2026-05-11 15:20:54,088 - INFO - Downloading OA package from https://ftp.ncbi.nlm.nih.gov/pub/pmc/oa_package/51/c2/PMC9749924.tar.gz
2026-05-11 15:20:54,826 - ERROR - Failed to download package: HTTP 404
2026-05-11 15:20:54,827 - INFO

{'session_start_time': '2026-05-11T15:20:50.354138',
 'completion_time': '2026-05-11T15:21:34.303381',
 'source_file': '/Users/yaochenr/project/pmc_data_mining/notebook/test_dois.csv',
 'output_directory': '/Users/yaochenr/project/pmc_data_mining/notebook/doi_mining_test',
 'paper_type': 'test',
 'statistics': {'total_dois': 8,
  'found_pmc_ids': 8,
  'successfully_processed': 7,
  'failed_processing': 0,
  'not_in_pmc': 0,
  'not_open_access': 0,
  'already_processed': 1},
 'successful_papers': [{'doi': '10.1039/d2md00199c', 'pmc_id': 'PMC9749924'},
  {'doi': '10.1002/anie.201913904', 'pmc_id': 'PMC7154537'},
  {'doi': '10.1007/s13238-018-0602-z', 'pmc_id': 'PMC6626596'},
  {'doi': '10.1007/s13238-020-00732-8', 'pmc_id': 'PMC7305269'},
  {'doi': '10.1021/acschembio.5b00216', 'pmc_id': 'PMC4548256'},
  {'doi': '10.1039/d2cb00223j', 'pmc_id': 'PMC9994103'},
  {'doi': '10.1021/acsmedchemlett.4c00250', 'pmc_id': 'PMC11472389'}],
 'failed_papers': []}

In [3]:
pmc_dirs = sorted(doi_out.glob("PMC*"))
print(f"Stored {len(pmc_dirs)} papers:")
for p in pmc_dirs:
    meta = json.loads((p / "metadata.json").read_text())
    n_images = len(list((p / "images").glob("*.jpg"))) if (p / "images").exists() else 0
    n_supp = len(list((p / "supplementary").iterdir())) if (p / "supplementary").exists() else 0
    print(f"  {p.name}  doi={meta.get('doi')}  images={n_images}  supp_files={n_supp}")

skipped = doi_out / "dois_notin_pmc.txt"
if skipped.exists():
    print("\nSkipped (not in PMC OA):")
    print(skipped.read_text())

Stored 7 papers:
  PMC11472389  doi=10.1021/acsmedchemlett.4c00250  images=3  supp_files=0
  PMC4548256  doi=10.1021/acschembio.5b00216  images=4  supp_files=0
  PMC6626596  doi=10.1007/s13238-018-0602-z  images=2  supp_files=0
  PMC7154537  doi=10.1002/anie.201913904  images=5  supp_files=0
  PMC7305269  doi=10.1007/s13238-020-00732-8  images=3  supp_files=0
  PMC9749924  doi=10.1039/d2md00199c  images=4  supp_files=0
  PMC9994103  doi=10.1039/d2cb00223j  images=5  supp_files=0
